In [7]:
import random


# ============================================================
# 1. USER INPUT
# ============================================================

def get_user_input():

    print("=" * 70)
    print("DOCTOR-HOSPITAL ASSIGNMENT SYSTEM")
    print("=" * 70)

    # Number of doctors
    while True:
        try:
            num_doctors = int(
                input("Enter number of doctors: ")
            )

            if num_doctors <= 0:
                print(
                    "Number of doctors must be positive."
                )
                continue

            break

        except ValueError:
            print(
                "Please enter a valid integer."
            )

    # Number of hospitals
    while True:
        try:
            num_hospitals = int(
                input("Enter number of hospitals: ")
            )

            if num_hospitals <= 0:
                print(
                    "Number of hospitals must be positive."
                )
                continue

            break

        except ValueError:
            print(
                "Please enter a valid integer."
            )

    # Create doctor names
    doctors = [
        f"D{i}"
        for i in range(1, num_doctors + 1)
    ]

    # Create hospital names
    hospitals = [
        f"H{i}"
        for i in range(1, num_hospitals + 1)
    ]

    # Hospital capacities
    capacities = {}

    print("\nEnter hospital capacities:")

    for hospital in hospitals:

        while True:

            try:
                capacity = int(
                    input(
                        f"Capacity for {hospital}: "
                    )
                )

                if capacity < 0:
                    print(
                        "Capacity cannot be negative."
                    )
                    continue

                capacities[hospital] = capacity

                break

            except ValueError:

                print(
                    "Please enter a valid integer."
                )

    # Check total capacity
    total_capacity = sum(
        capacities.values()
    )

    print(
        f"\nTotal hospital capacity = "
        f"{total_capacity}"
    )

    print(
        f"Number of doctors = "
        f"{num_doctors}"
    )

    if total_capacity < num_doctors:

        raise ValueError(
            "\nERROR: Total hospital capacity "
            "must be greater than or equal to "
            "the number of doctors."
        )

    print(
        "\nCapacity requirement satisfied."
    )

    return (
        doctors,
        hospitals,
        capacities
    )


# ============================================================
# 2. RANDOMLY GENERATE PREFERENCES
# ============================================================

def generate_random_preferences(
    doctors,
    hospitals
):

    preferences = {}

    for doctor in doctors:

        # random.sample creates a random permutation
        preferences[doctor] = random.sample(
            hospitals,
            len(hospitals)
        )

    return preferences


# ============================================================
# 3. PRINT PREFERENCE TABLE
# ============================================================

def print_preferences(
    doctors,
    preferences
):

    print("\n")
    print("=" * 70)
    print("RANDOMLY GENERATED PREFERENCE TABLE")
    print("=" * 70)

    for doctor in doctors:

        preference_string = " > ".join(
            preferences[doctor]
        )

        print(
            f"{doctor:<6}: "
            f"{preference_string}"
        )


# ============================================================
# 4. CREATE HOSPITAL SLOTS
# ============================================================

def create_slots(
    capacities
):

    slots = []

    for hospital, capacity in capacities.items():

        for slot_number in range(
            1,
            capacity + 1
        ):

            slots.append(
                {
                    "hospital": hospital,
                    "slot": (
                        f"{hospital}_"
                        f"{slot_number}"
                    )
                }
            )

    return slots


# ============================================================
# 5. CREATE RANK LOOKUP
# ============================================================

def create_rank_lookup(
    preferences
):

    rank_lookup = {}

    for doctor, preference_list in (
        preferences.items()
    ):

        rank_lookup[doctor] = {}

        for rank, hospital in enumerate(
            preference_list,
            start=1
        ):

            rank_lookup[doctor][hospital] = rank

    return rank_lookup


# ============================================================
# 6. BUILD COST MATRIX
# ============================================================

def build_cost_matrix(
    doctors,
    slots,
    rank_lookup
):

    cost_matrix = []

    for doctor in doctors:

        row = []

        for slot in slots:

            hospital = slot[
                "hospital"
            ]

            cost = rank_lookup[
                doctor
            ][hospital]

            row.append(cost)

        cost_matrix.append(row)

    return cost_matrix


# ============================================================
# 7. PRINT COST MATRIX
# ============================================================

def print_cost_matrix(
    doctors,
    slots,
    cost_matrix
):

    print("\n")
    print("=" * 70)
    print("COST MATRIX")
    print("=" * 70)

    slot_names = [
        slot["slot"]
        for slot in slots
    ]

    header = (
        f"{'Doctor':<8}"
    )

    for slot in slot_names:

        header += (
            f"{slot:>6}"
        )

    print(header)

    print(
        "-" * len(header)
    )

    for doctor, row in zip(
        doctors,
        cost_matrix
    ):

        line = (
            f"{doctor:<8}"
        )

        for cost in row:

            line += (
                f"{cost:>6}"
            )

        print(line)


# ============================================================
# 8. HUNGARIAN ALGORITHM
# ============================================================

def hungarian_algorithm(
    cost_matrix
):

    n = len(
        cost_matrix
    )

    m = len(
        cost_matrix[0]
    )

    if n > m:

        raise ValueError(
            "Number of hospital slots "
            "must be >= number of doctors."
        )

    u = [0] * (
        n + 1
    )

    v = [0] * (
        m + 1
    )

    p = [0] * (
        m + 1
    )

    way = [0] * (
        m + 1
    )

    for i in range(
        1,
        n + 1
    ):

        p[0] = i

        j0 = 0

        minv = [
            float("inf")
        ] * (
            m + 1
        )

        used = [
            False
        ] * (
            m + 1
        )

        while True:

            used[j0] = True

            i0 = p[j0]

            delta = (
                float("inf")
            )

            j1 = 0

            for j in range(
                1,
                m + 1
            ):

                if not used[j]:

                    current_cost = (
                        cost_matrix[
                            i0 - 1
                        ][
                            j - 1
                        ]
                        - u[i0]
                        - v[j]
                    )

                    if (
                        current_cost
                        < minv[j]
                    ):

                        minv[j] = (
                            current_cost
                        )

                        way[j] = j0

                    if (
                        minv[j]
                        < delta
                    ):

                        delta = minv[j]

                        j1 = j

            for j in range(
                0,
                m + 1
            ):

                if used[j]:

                    u[
                        p[j]
                    ] += delta

                    v[j] -= delta

                else:

                    minv[j] -= delta

            j0 = j1

            if p[j0] == 0:

                break

        while True:

            j1 = way[j0]

            p[j0] = p[j1]

            j0 = j1

            if j0 == 0:

                break

    assignment = [
        -1
    ] * n

    for j in range(
        1,
        m + 1
    ):

        if p[j] != 0:

            doctor_index = (
                p[j] - 1
            )

            slot_index = (
                j - 1
            )

            assignment[
                doctor_index
            ] = slot_index

    return assignment


# ============================================================
# 9. HUNGARIAN ASSIGNMENT
# ============================================================

def get_hungarian_assignment(
    doctors,
    slots,
    cost_matrix
):

    slot_indices = (
        hungarian_algorithm(
            cost_matrix
        )
    )

    assignment = {}

    for (
        doctor_index,
        slot_index
    ) in enumerate(
        slot_indices
    ):

        doctor = doctors[
            doctor_index
        ]

        slot = slots[
            slot_index
        ]

        assignment[
            doctor
        ] = {
            "hospital":
                slot["hospital"],

            "slot":
                slot["slot"]
        }

    return assignment


# ============================================================
# 10. GREEDY BASELINE
# ============================================================

def greedy_assignment(
    doctors,
    preferences,
    capacities
):

    remaining_capacity = (
        capacities.copy()
    )

    assignment = {}

    for doctor in doctors:

        for hospital in (
            preferences[doctor]
        ):

            if (
                remaining_capacity[
                    hospital
                ]
                > 0
            ):

                assignment[
                    doctor
                ] = {
                    "hospital":
                        hospital
                }

                remaining_capacity[
                    hospital
                ] -= 1

                break

    return assignment


# ============================================================
# 11. EVALUATION
# ============================================================

def evaluate_assignment(
    assignment,
    rank_lookup
):

    total_cost = 0

    first_choice_count = 0

    assigned_ranks = {}

    for (
        doctor,
        result
    ) in assignment.items():

        hospital = result[
            "hospital"
        ]

        rank = (
            rank_lookup[
                doctor
            ][hospital]
        )

        assigned_ranks[
            doctor
        ] = rank

        total_cost += rank

        if rank == 1:

            first_choice_count += 1

    num_doctors = len(
        assignment
    )

    average_rank = (
        total_cost
        / num_doctors
    )

    first_choice_rate = (
        first_choice_count
        / num_doctors
    )

    return {
        "total_cost":
            total_cost,

        "average_rank":
            average_rank,

        "first_choice_count":
            first_choice_count,

        "first_choice_rate":
            first_choice_rate,

        "assigned_ranks":
            assigned_ranks
    }


# ============================================================
# 12. PRINT RESULTS
# ============================================================

def print_results(
    title,
    doctors,
    assignment,
    preferences,
    metrics
):

    print("\n")
    print("=" * 70)
    print(title)
    print("=" * 70)

    print(
        f"{'Doctor':<8}"
        f"{'Assigned':<12}"
        f"{'Rank':<8}"
        f"{'Preference List'}"
    )

    print(
        "-" * 70
    )

    for doctor in doctors:

        hospital = (
            assignment[
                doctor
            ][
                "hospital"
            ]
        )

        rank = (
            metrics[
                "assigned_ranks"
            ][doctor]
        )

        pref_string = (
            " > ".join(
                preferences[
                    doctor
                ]
            )
        )

        print(
            f"{doctor:<8}"
            f"{hospital:<12}"
            f"{rank:<8}"
            f"{pref_string}"
        )

    print("\nMetrics")
    print("-" * 30)

    print(
        "Total rank cost:",
        metrics[
            "total_cost"
        ]
    )

    print(
        "Average rank:",
        round(
            metrics[
                "average_rank"
            ],
            3
        )
    )

    print(
        "First-choice count:",
        metrics[
            "first_choice_count"
        ]
    )

    print(
        "First-choice rate:",
        f"{metrics['first_choice_rate'] * 100:.2f}%"
    )


# ============================================================
# 13. COMPARE RESULTS
# ============================================================

def compare_results(
    hungarian_metrics,
    greedy_metrics
):

    print("\n")
    print("=" * 70)
    print("HUNGARIAN VS GREEDY")
    print("=" * 70)

    print(
        f"{'Metric':<25}"
        f"{'Hungarian':<20}"
        f"{'Greedy':<20}"
    )

    print("-" * 65)

    print(
        f"{'Total Rank Cost':<25}"
        f"{hungarian_metrics['total_cost']:<20}"
        f"{greedy_metrics['total_cost']:<20}"
    )

    print(
        f"{'Average Rank':<25}"
        f"{hungarian_metrics['average_rank']:<20.3f}"
        f"{greedy_metrics['average_rank']:<20.3f}"
    )

    h_rate = (
        hungarian_metrics[
            "first_choice_rate"
        ]
        * 100
    )

    g_rate = (
        greedy_metrics[
            "first_choice_rate"
        ]
        * 100
    )

    print(
        f"{'First Choice Rate':<25}"
        f"{h_rate:<19.2f}%"
        f"{g_rate:<19.2f}%"
    )


# ============================================================
# 14. MAIN
# ============================================================

def main():

    # ----------------------------------------
    # User input
    # ----------------------------------------

    (
        doctors,
        hospitals,
        capacities
    ) = get_user_input()

    # ----------------------------------------
    # Random preference generation
    # ----------------------------------------

    preferences = (
        generate_random_preferences(
            doctors,
            hospitals
        )
    )

    # Print preferences
    print_preferences(
        doctors,
        preferences
    )

    # ----------------------------------------
    # Create slots
    # ----------------------------------------

    slots = create_slots(
        capacities
    )

    print("\nHospital slots:")

    print(
        [
            slot["slot"]
            for slot in slots
        ]
    )

    # ----------------------------------------
    # Rank lookup
    # ----------------------------------------

    rank_lookup = (
        create_rank_lookup(
            preferences
        )
    )

    # ----------------------------------------
    # Cost matrix
    # ----------------------------------------

    cost_matrix = (
        build_cost_matrix(
            doctors,
            slots,
            rank_lookup
        )
    )

    print_cost_matrix(
        doctors,
        slots,
        cost_matrix
    )

    # ----------------------------------------
    # Hungarian
    # ----------------------------------------

    hungarian_result = (
        get_hungarian_assignment(
            doctors,
            slots,
            cost_matrix
        )
    )

    hungarian_metrics = (
        evaluate_assignment(
            hungarian_result,
            rank_lookup
        )
    )

    # ----------------------------------------
    # Greedy
    # ----------------------------------------

    greedy_result = (
        greedy_assignment(
            doctors,
            preferences,
            capacities
        )
    )

    greedy_metrics = (
        evaluate_assignment(
            greedy_result,
            rank_lookup
        )
    )

    # ----------------------------------------
    # Print results
    # ----------------------------------------

    print_results(
        "HUNGARIAN ALGORITHM",
        doctors,
        hungarian_result,
        preferences,
        hungarian_metrics
    )

    print_results(
        "GREEDY BASELINE",
        doctors,
        greedy_result,
        preferences,
        greedy_metrics
    )

    compare_results(
        hungarian_metrics,
        greedy_metrics
    )


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":
    main()

DOCTOR-HOSPITAL ASSIGNMENT SYSTEM


Enter number of doctors:  18
Enter number of hospitals:  6



Enter hospital capacities:


Capacity for H1:  4
Capacity for H2:  2
Capacity for H3:  5
Capacity for H4:  4
Capacity for H5:  1
Capacity for H6:  3



Total hospital capacity = 19
Number of doctors = 18

Capacity requirement satisfied.


RANDOMLY GENERATED PREFERENCE TABLE
D1    : H2 > H5 > H3 > H4 > H6 > H1
D2    : H6 > H4 > H5 > H2 > H1 > H3
D3    : H1 > H3 > H5 > H2 > H4 > H6
D4    : H6 > H2 > H1 > H5 > H4 > H3
D5    : H5 > H3 > H6 > H2 > H4 > H1
D6    : H4 > H6 > H1 > H3 > H2 > H5
D7    : H5 > H6 > H3 > H1 > H2 > H4
D8    : H4 > H5 > H1 > H3 > H2 > H6
D9    : H6 > H4 > H5 > H3 > H1 > H2
D10   : H1 > H4 > H2 > H3 > H6 > H5
D11   : H2 > H6 > H1 > H4 > H5 > H3
D12   : H5 > H1 > H3 > H4 > H6 > H2
D13   : H1 > H6 > H2 > H4 > H5 > H3
D14   : H6 > H2 > H3 > H4 > H5 > H1
D15   : H1 > H5 > H3 > H6 > H4 > H2
D16   : H3 > H5 > H6 > H2 > H1 > H4
D17   : H6 > H5 > H2 > H1 > H3 > H4
D18   : H3 > H1 > H6 > H2 > H4 > H5

Hospital slots:
['H1_1', 'H1_2', 'H1_3', 'H1_4', 'H2_1', 'H2_2', 'H3_1', 'H3_2', 'H3_3', 'H3_4', 'H3_5', 'H4_1', 'H4_2', 'H4_3', 'H4_4', 'H5_1', 'H6_1', 'H6_2', 'H6_3']


COST MATRIX
Doctor    H1_1  H1_2  H1_3  H1_4  H2_1  H2_2